## 1. Installation

In [ ]:
!pip install -q -U transformers peft accelerate bitsandbytes datasets
!pip install -q unsloth

## 2. Imports, configuration, reproductibilité

In [ ]:
import json, math, os, random, re, time, hashlib
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd, torch

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DATA_DIR = Path("outputs/step3_dataset")
OUT_DIR = Path("outputs/step4_qlora"); OUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"                # base en 4-bit NF4 chargée à la section 4
dataset_cfg = json.load(open(DATA_DIR / "dataset_config.json", encoding="utf-8"))
MAX_SEQ_LENGTH = int(dataset_cfg["stats"]["max_seq_length"])
rag_prompt = json.load(open(DATA_DIR / "rag_prompt.json", encoding="utf-8"))
ABSTAIN_MARKER = rag_prompt["abstain_marker"]

HP = dict(lora_r=16, lora_alpha=16, lora_dropout=0.0, lr=2e-4, epochs=2, per_device_bs=1, grad_accum=16, warmup_ratio=0.05, weight_decay=0.0)
assert torch.cuda.is_available(), "GPU requis (Colab : Exécution > Modifier le type d'exécution > GPU)"
gpu = torch.cuda.get_device_properties(0)
BF16 = torch.cuda.get_device_capability()[0] >= 8      # T4 (7.5) : bf16 seulement émulé -> fp16 ; torch.cuda.is_bf16_supported() peut répondre True à tort
print(f"GPU : {gpu.name} ({gpu.total_memory / 1e9:.0f} Go) | bf16 : {BF16} | max_seq_length : {MAX_SEQ_LENGTH} | k_context étape 3 : {dataset_cfg['k_context']}")